# Plan #28 Iteration Visibility

This notebook is a visibility surface for the current DIGIMON iteration loop.

What it is good for:
- comparing benchmark artifacts across runs
- inspecting question-level regressions
- seeing what changed between the anchor-guard and helper-fallback phases
- making the next gate decision explicit before spending on the 19q lane again

What it is not:
- a live trace of the agent's internal reasoning
- a perfect source of truth for nested helper fallbacks

Current limitation: the April 3, 2026 helper-fallback smoke run showed helper fallback events in stderr, but the benchmark JSON still recorded `fallback_used_any=false`. That observability gap is part of the current blocker.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from textwrap import shorten

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS = ROOT / 'results'
STATUS_PATH = ROOT / 'CURRENT_STATUS.md'
PLAN_PATH = ROOT / 'docs' / 'plans' / '28_truthful_overnight_stabilization_and_contract_repair.md'

RUN_FILES = [
    RESULTS / 'MuSiQue_gpt-5-4-mini_consolidated_20260403T055717Z.json',
    RESULTS / 'MuSiQue_gpt-5-4-mini_consolidated_20260403T130547Z.json',
    RESULTS / 'MuSiQue_gpt-5-4-mini_consolidated_20260403T131543Z.json',
    RESULTS / 'MuSiQue_gpt-5-4-mini_consolidated_20260403T133705Z.json',
]

QUESTION_ID = '2hop__619265_45326'

ROOT, RESULTS.exists(), STATUS_PATH.exists(), PLAN_PATH.exists()

In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


def render_table(rows: list[dict], columns: list[str]) -> str:
    widths = {col: len(col) for col in columns}
    for row in rows:
        for col in columns:
            widths[col] = max(widths[col], len(str(row.get(col, ''))))
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    divider = '-+-'.join('-' * widths[col] for col in columns)
    body = [
        ' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns)
        for row in rows
    ]
    return '\n'.join([header, divider, *body])


def summarize_run(path: Path) -> dict:
    data = load_json(path)
    return {
        'run': path.name.replace('.json', ''),
        'n_q': data.get('n_questions'),
        'done': data.get('n_completed'),
        'em': data.get('avg_em'),
        'llm_em': data.get('avg_llm_em_judged'),
        'f1': data.get('avg_f1'),
        'cost': data.get('total_cost'),
        'fallback_rate': data.get('fallback_usage_rate_any'),
        'provider_fail_rate': data.get('provider_failure_rate'),
        'retrieval_stagnation_rate': data.get('retrieval_stagnation_rate'),
    }


def question_result(path: Path, question_id: str) -> dict | None:
    data = load_json(path)
    for result in data.get('results', []):
        if result.get('id') == question_id:
            return result
    return None


def question_snapshot(path: Path, question_id: str) -> dict:
    result = question_result(path, question_id)
    if result is None:
        return {
            'run': path.name.replace('.json', ''),
            'present': False,
        }
    composability = result.get('composability', {}) or {}
    error_examples = composability.get('error_examples', []) or []
    first_error = error_examples[0].get('error') if error_examples else ''
    return {
        'run': path.name.replace('.json', ''),
        'present': True,
        'predicted': result.get('predicted'),
        'gold': result.get('gold'),
        'em': result.get('em'),
        'llm_em': result.get('llm_em'),
        'f1': result.get('f1'),
        'n_tool_calls': result.get('n_tool_calls'),
        'fallback_used_any': result.get('fallback_used_any'),
        'model_fallback_used': result.get('model_fallback_used'),
        'failure_codes': ','.join(result.get('failure_event_codes', []) or []),
        'first_error': first_error,
        'reasoning': result.get('reasoning', ''),
    }


In [ ]:
run_rows = [summarize_run(path) for path in RUN_FILES if path.exists()]
print(render_table(run_rows, [
    'run', 'n_q', 'done', 'em', 'llm_em', 'f1', 'cost', 'fallback_rate',
    'provider_fail_rate', 'retrieval_stagnation_rate'
]))

In [ ]:
question_rows = [question_snapshot(path, QUESTION_ID) for path in RUN_FILES if path.exists()]
print(render_table(question_rows, [
    'run', 'present', 'predicted', 'gold', 'em', 'llm_em', 'f1',
    'n_tool_calls', 'fallback_used_any', 'model_fallback_used', 'failure_codes'
]))

for row in question_rows:
    print('\n' + '=' * 120)
    print(row['run'])
    print('- reasoning:', row.get('reasoning', ''))
    print('- first error:', row.get('first_error', ''))

## What "Tune A Dedicated Helper Fallback Order" Means

DIGIMON has more than one model lane:

- primary benchmark model: the agent that decides what tools to call and when to submit
- helper model lane: small structured helper calls like semantic-plan revise, atom completion, bridge judging, and related internal checks

The problem is that a helper fallback is not neutral. If Gemini helper calls fail and the helper lane falls back to a different model, that different model can change:

- whether an atom is judged complete
- whether a bridge candidate is accepted
- whether a semantic-plan revise pass changes scope or leaves the draft alone

So "tune a dedicated helper fallback order" means:

1. choose a specific ordered backup list just for helper calls
2. test that order on a few known IDs where helper behavior matters
3. lock the order only after it is both stable and quality-preserving

Example candidate helper orders:

- `gemini/gemini-2.5-flash -> openrouter/openai/gpt-5.4-mini -> deepseek/deepseek-chat`
- `gemini/gemini-2.5-flash -> deepseek/deepseek-chat -> openrouter/openai/gpt-5.4-mini`
- skip Gemini for helpers entirely if quota instability keeps dominating the lane

Why test on a few known IDs first instead of the whole 19q set:

- it is cheaper
- it isolates whether the helper lane itself is changing answers
- it prevents the 19q gate from mixing model-routing noise with controller progress

In [ ]:
candidate_helper_orders = [
    ['gemini/gemini-2.5-flash', 'openrouter/openai/gpt-5.4-mini', 'deepseek/deepseek-chat'],
    ['gemini/gemini-2.5-flash', 'deepseek/deepseek-chat', 'openrouter/openai/gpt-5.4-mini'],
    ['openrouter/openai/gpt-5.4-mini', 'deepseek/deepseek-chat'],
]

known_ids = [
    '2hop__619265_45326',
    '2hop__199513_792744',
    '2hop__354635_555659',
    '2hop__820301_510149',
]

for order in candidate_helper_orders:
    primary = order[0]
    fallbacks = order[1:]
    print('\nORDER:', ' -> '.join(order))
    print('Recommended smoke-test pattern:')
    print(
        'DIGIMON_AGENTIC_MODEL={primary} '.format(primary=primary)
        + 'python scripts/run_with_digimon_python.py eval/run_agent_benchmark.py '
        + '--agent-spec none --allow-missing-agent-spec --missing-agent-spec-reason relocated '
        + '--dataset MuSiQue --data-root ~/projects/Digimon_for_KG_application/Data '
        + '--questions {qid} --model openrouter/openai/gpt-5.4-mini --backend direct '
        + '--retrieval-stagnation-turns 6 --question-delay 0 --tag helper_order_smoke'
    )
    print('Expected manual checks: predicted answer, helper regressions, fallback visibility, composability errors')

print('\nKnown probe IDs:', ', '.join(known_ids))

In [ ]:
status_text = STATUS_PATH.read_text() if STATUS_PATH.exists() else ''
plan_text = PLAN_PATH.read_text() if PLAN_PATH.exists() else ''

for title, text in [('CURRENT_STATUS.md', status_text), ('Plan 28', plan_text)]:
    print('\n' + '=' * 120)
    print(title)
    print('=' * 120)
    for needle in [
        'Validate helper fallback quality + observability before spending on the 19q gate',
        'helper structured calls',
        'results/MuSiQue_gpt-5-4-mini_consolidated_20260403T133705Z.json',
    ]:
        idx = text.find(needle)
        if idx >= 0:
            snippet = text[max(0, idx - 180): idx + 520]
            print('\n--- snippet for:', needle)
            print(snippet)
